# Member 1 — Exact CVRP Method | Version 2: CP-SAT Model
**Git commit:** `feat(exact): implement CP-SAT multiple-circuit CVRP model`

This notebook implements an arc-based exact model with OR-Tools CP-SAT. It minimizes total rounded Euclidean distance, forces every customer to be visited once, limits the number of routes, and propagates vehicle loads to enforce capacity and prevent customer-only subtours.

## 1. Install dependencies and create the shared instance

In [ ]:
!pip -q install numpy pandas matplotlib ortools

In [ ]:
import math, random, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
N_CUSTOMERS = 20   # Increase to 31 to match the scale of the sample report
K = 5
CAPACITY = 100
rng = np.random.default_rng(SEED)

# Shared deterministic synthetic instance. Both members must use these same parameters.
coords = [(50.0, 50.0)] + [(float(x), float(y)) for x,y in rng.uniform(0,100,size=(N_CUSTOMERS,2))]
demands = [0] + [int(x) for x in rng.integers(4, 11, size=N_CUSTOMERS)]
n = len(coords)
dist = np.zeros((n,n), dtype=int)
for i in range(n):
    for j in range(n):
        if i != j:
            dist[i,j] = int(math.floor(math.dist(coords[i], coords[j]) + 0.5))

print(f"Nodes: {n} (including depot 0); customers: {N_CUSTOMERS}; vehicles: {K}; capacity: {CAPACITY}")
print(f"Total demand: {sum(demands)}; total fleet capacity: {K*CAPACITY}")
assert sum(demands) <= K*CAPACITY, "Infeasible generated instance; adjust seed/demands."

## 2. Build and solve the exact model

In [ ]:
from ortools.sat.python import cp_model

def solve_cvrp_cpsat(time_limit_s=60, workers=8):
    model = cp_model.CpModel()
    # x[i,j] indicates that an arc is used. AddMultipleCircuit allows multiple depot tours.
    x = {}
    arcs = []
    for i in range(n):
        for j in range(n):
            if i != j:
                x[i,j] = model.NewBoolVar(f"x_{i}_{j}")
                arcs.append((i,j,x[i,j]))
    # AddMultipleCircuit requires each non-depot node to be visited in a circuit;
    # multiple tours can leave and return to depot 0.
    model.AddMultipleCircuit(arcs)

    # Load after serving each customer. Conditional load propagation eliminates customer subtours.
    load = {}
    for i in range(1,n):
        load[i] = model.NewIntVar(demands[i], CAPACITY, f"load_{i}")
    for i in range(1,n):
        for j in range(1,n):
            if i != j:
                model.Add(load[j] >= load[i] + demands[j]).OnlyEnforceIf(x[i,j])

    # Fleet limit: count arcs leaving depot.
    model.Add(sum(x[0,j] for j in range(1,n)) <= K)
    model.Minimize(sum(int(dist[i,j]) * x[i,j] for (i,j) in x))

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = float(time_limit_s)
    solver.parameters.num_search_workers = int(workers)
    solver.parameters.random_seed = SEED
    start = time.perf_counter()
    status = solver.Solve(model)
    elapsed = time.perf_counter() - start

    status_name = solver.StatusName(status)
    result = {"status": status_name, "runtime_s": elapsed, "objective": None,
              "best_bound": None, "routes": [], "optimality_proven": status == cp_model.OPTIMAL}
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        succ = {}
        for (i,j), var in x.items():
            if solver.Value(var):
                succ.setdefault(i, []).append(j)
        # Follow selected arcs from the depot; each route starts with a depot outgoing arc.
        remaining = set(range(1,n))
        routes = []
        for first in succ.get(0, []):
            route, current = [], first
            while current != 0 and current not in route:
                route.append(current)
                remaining.discard(current)
                nxt = succ.get(current, [])
                if not nxt: break
                current = nxt[0]
            if route:
                routes.append(route)
        # Defensive diagnostic: should be empty for a valid AddMultipleCircuit solution.
        if remaining:
            print("Warning: customers not extracted from selected arcs:", sorted(remaining))
        result.update(objective=int(round(solver.ObjectiveValue())),
                      best_bound=float(solver.BestObjectiveBound()), routes=routes)
        result["validation"] = validate_routes(routes)
    print("Status:", status_name)
    print("Runtime (s):", round(elapsed,3))
    if result["objective"] is not None:
        print("Objective:", result["objective"], "| best bound:", round(result["best_bound"],2),
              "| optimality proven:", result["optimality_proven"])
    return result

exact_result = solve_cvrp_cpsat(time_limit_s=60)

## 3. Interpretation
- `OPTIMAL` means CP-SAT proved the objective optimal.
- `FEASIBLE` means a valid incumbent was found, but optimality was not proven within the time limit.
- `UNKNOWN` means no usable solution was returned within the limit.
- Never label a time-limited incumbent as proven optimal.
- Save the solver status, runtime, objective, best bound, and independent feasibility result for the report.